In [ ]:
import torch
import torchvision.transforms as T
from torch.utils.data import IterableDataset
from PIL import Image

class CLIPDataset(IterableDataset):
    def __init__(self, stream, tokenizer, max_seq_len=512, length=100000):
        self.stream = stream
        self.tokenizer = tokenizer
        self.max_seq_len = max_seq_len
        self.transform = T.Compose([
            T.Resize((224,224)),
            T.RandomHorizontalFlip(p=0.5),
            T.ToTensor(),
            T.Normalize(
                mean=(0.48145466, 0.4578275, 0.40821073),
                std=(0.26862954, 0.26130258, 0.27577711)
            )
        ])
        self.length = length

    def __len__(self):
      return self.length

    def __iter__(self):
      for sample in self.stream:
        image = sample["image"]
        text = sample["caption"]

        try:
          if not isinstance(image, Image.Image):
            image = Image.open(image).convert("RGB")

          image = self.transform(image)

          encoding = self.tokenizer(
              text,
              padding="max_length",
              truncation = True,
              max_length=self.max_seq_len,
              return_tensors="pt"
          )

          input_ids = encoding['input_ids'].squeeze(0)
          attention_mask = encoding['attention_mask'].squeeze(0)

          yield image, input_ids, attention_mask

        except Exception:
          continue